# 漫画图像转视频配音系统

本notebook实现了一个完整的流程：
1. 使用Bedrock Nova多模态模型提取漫画图像关键内容
2. 调用ComfyUI接口进行图生视频
3. 通过GPT-SoVITS接口生成语音
4. 合并视频和音频

## 特点
- 🖼️ **批量处理**: 支持目录下多张漫画图像的批量处理
- 🎬 **图生视频**: 使用ComfyUI生成动态视频
- 🔊 **语音合成**: 使用GPT-SoVITS生成高质量语音
- 🎞️ **视频合成**: 自动合并多段视频和音频

## 1. 环境设置和导入

In [1]:
import base64
import boto3
import json
import time
import os
import requests
import urllib.request
import urllib.parse
import uuid
import subprocess
import io
import wave
from datetime import datetime
from typing import Dict, List, Tuple
from pathlib import Path
import numpy as np
from pydub import AudioSegment

# 创建必要的目录
os.makedirs('input_images', exist_ok=True)
os.makedirs('output_videos', exist_ok=True)
os.makedirs('output_audio', exist_ok=True)
os.makedirs('final_videos', exist_ok=True)
os.makedirs('temp', exist_ok=True)

print("✅ 环境设置完成")
print("✅ 目录结构创建完成")

✅ 环境设置完成
✅ 目录结构创建完成


## 2. 配置参数

In [ ]:
# Bedrock配置
BEDROCK_REGION = "us-west-2"
BEDROCK_MODEL_ID = "us.amazon.nova-lite-v1:0"

# ComfyUI配置 (用户需要后续配置)
COMFYUI_SERVER_URL = "http://ec2-35-84-2-12.us-west-2.compute.amazonaws.com:8188"  # 用户需要填入ComfyUI服务器地址
COMFYUI_WORKFLOW_PATH = "./sample_workflow.json"  # 用户需要填入workflow JSON文件路径

# GPT-SoVITS配置
GPT_SOVITS_ENDPOINT = "gpt-sovits-inference-2025-08-12-07-51-09-161"  # 用户需要填入GPT-SoVITS endpoint名称
REFERENCE_AUDIO_PATH = "s3://sagemaker-us-west-2-687912291502/gpt-sovits/wav/speech_20240425104005663.mp3"  # 用户需要填入参考音频路径
REFERENCE_TEXT = "它包括以下几个主要方面:SAP系统管理包括SAP系统实例的安装、启动、监控、备份、升级等日常管理任务。Basis团队负责保证系统的正常运行。"

# 批处理配置
BATCH_SIZE = 3  # 每批处理的图像数量
MAX_IMAGES = 10  # 最大处理图像数量

print("⚠️ 请确保在运行前配置好以下参数:")
print("- COMFYUI_SERVER_URL")
print("- COMFYUI_WORKFLOW_PATH")
print("- GPT_SOVITS_ENDPOINT")
print("- REFERENCE_AUDIO_PATH")
print("- REFERENCE_TEXT")

## 3. Bedrock Nova 图像内容提取功能

In [ ]:
# 创建Bedrock客户端
bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=BEDROCK_REGION,
)

def analyze_comic_image_content(image_path: str, batch_index: int = 0) -> Dict:
    """
    使用Bedrock Nova模型分析漫画图像内容，提取关键信息用于视频生成
    
    Args:
        image_path: 图像文件路径
        batch_index: 批次索引
    
    Returns:
        包含分析结果的字典
    """
    start_time = time.time()
    
    try:
        # 读取图像文件并编码为Base64
        with open(image_path, "rb") as image_file:
            binary_data = image_file.read()
            base_64_encoded_data = base64.b64encode(binary_data)
            base64_string = base_64_encoded_data.decode("utf-8")
        
        # 漫画内容分析提示词
        custom_prompt = """请仔细分析这张漫画图像，并以JSON格式返回以下信息（使用中文回答）：
{
  "scene_description": "场景的详细描述，包括背景、环境、氛围",
  "characters": "人物描述，包括外观、表情、动作、服装",
  "dialogue_text": "图中的对话文字或旁白文字（如果有的话）",
  "story_content": "这一格漫画想要表达的故事内容或情节",
  "visual_style": "画面风格描述，如色彩、线条、构图等",
  "emotion_tone": "整体情感基调（如欢快、紧张、温馨、悲伤等）",
  "video_prompt": "适合用于图生视频的英文提示词，描述如何让这个画面动起来",
  "audio_script": "适合配音的文本内容，可以是对话、旁白或场景描述",
  "key_elements": "画面中的关键元素列表"
}"""
        
        # 定义系统提示
        system_list = [
            {
                "text": "你是一个专业的漫画分析师和视频制作专家，擅长从漫画图像中提取关键信息并转化为视频制作素材。请客观、详细地分析漫画内容。"
            }
        ]
        
        # 定义用户消息
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",
                            "source": {"bytes": base64_string},
                        }
                    },
                    {
                        "text": custom_prompt
                    },
                ],
            }
        ]
        
        # 配置推理参数
        inf_params = {"maxTokens": 800, "topP": 0.1, "topK": 20, "temperature": 0.3}
        
        native_request = {
            "schemaVersion": "messages-v1",
            "messages": message_list,
            "system": system_list,
            "inferenceConfig": inf_params,
        }
        
        # 调用Bedrock API
        api_start_time = time.time()
        response = bedrock_client.invoke_model(modelId=BEDROCK_MODEL_ID, body=json.dumps(native_request))
        model_response = json.loads(response["body"].read())
        api_end_time = time.time()
        
        # 计算时延
        total_latency = api_end_time - start_time
        api_latency = api_end_time - api_start_time
        
        # 提取响应内容
        content_text = model_response["output"]["message"]["content"][0]["text"]
        
        # 尝试解析JSON响应
        try:
            parsed_content = json.loads(content_text)
        except json.JSONDecodeError:
            # 如果不是有效JSON，创建结构化响应
            parsed_content = {
                "scene_description": content_text,
                "characters": "未识别",
                "dialogue_text": "",
                "story_content": content_text,
                "visual_style": "未识别",
                "emotion_tone": "未识别",
                "video_prompt": "animate the comic scene",
                "audio_script": content_text[:200],
                "key_elements": [],
                "raw_response": content_text
            }
        
        return {
            'success': True,
            'file_path': image_path,
            'batch_index': batch_index,
            'analysis_result': parsed_content,
            'raw_response': content_text,
            'model_id': BEDROCK_MODEL_ID,
            'usage': model_response.get('usage', {}),
            'latency': {
                'total_ms': round(total_latency * 1000, 2),
                'api_ms': round(api_latency * 1000, 2)
            },
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'success': False,
            'file_path': image_path,
            'batch_index': batch_index,
            'error': str(e),
            'latency': {
                'total_ms': round((time.time() - start_time) * 1000, 2),
                'api_ms': 0
            },
            'timestamp': datetime.now().isoformat()
        }

print("✅ Bedrock Nova 图像分析功能已定义")

## 4. 批量图像处理功能

In [ ]:
def get_comic_images_from_directory(directory_path: str) -> List[str]:
    """
    从目录中获取所有支持的图像文件
    
    Args:
        directory_path: 目录路径
    
    Returns:
        图像文件路径列表
    """
    supported_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    
    image_files = []
    
    if not os.path.exists(directory_path):
        print(f"❌ 目录不存在: {directory_path}")
        return image_files
    
    print(f"📂 扫描目录: {directory_path}")
    
    for file in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path):
            file_ext = os.path.splitext(file)[1].lower()
            if file_ext in supported_extensions:
                image_files.append(file_path)
                print(f"  ✅ 找到图像: {file}")
    
    print(f"📊 总共找到 {len(image_files)} 个图像文件")
    return sorted(image_files)

def batch_analyze_comic_images(directory_path: str, max_files: int = None) -> List[Dict]:
    """
    批量分析目录中的漫画图像
    
    Args:
        directory_path: 目录路径
        max_files: 最大处理文件数
    
    Returns:
        分析结果列表
    """
    # 获取目录中的所有图像文件
    image_files = get_comic_images_from_directory(directory_path)
    
    if not image_files:
        print("❌ 目录中没有找到支持的图像文件")
        return []
    
    # 限制处理文件数量
    if max_files and len(image_files) > max_files:
        print(f"⚠️ 文件数量超过限制，只处理前 {max_files} 个文件")
        image_files = image_files[:max_files]
    
    results = []
    key_images_info = []  # 记录包含关键信息的图像索引和信息
    total_start_time = time.time()
    
    print(f"\n🚀 开始批量分析 {len(image_files)} 个图像...\n")
    
    # 按批次处理
    for batch_start in range(0, len(image_files), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(image_files))
        batch_files = image_files[batch_start:batch_end]
        
        print(f"📦 处理批次 {batch_start//BATCH_SIZE + 1}: 图像 {batch_start+1}-{batch_end}")
        
        batch_results = []
        for i, file_path in enumerate(batch_files):
            global_index = batch_start + i
            relative_path = os.path.relpath(file_path, directory_path)
            print(f"📁 处理图像 {global_index+1}/{len(image_files)}: {relative_path}")
            
            try:
                result = analyze_comic_image_content(file_path, global_index)
                batch_results.append(result)
                
                # 检查是否包含关键信息
                if result.get('success') and result.get('analysis_result'):
                    analysis = result['analysis_result']
                    # 如果有对话文字或故事内容，认为是关键图像
                    if (analysis.get('dialogue_text') and analysis['dialogue_text'].strip()) or \
                       (analysis.get('story_content') and len(analysis['story_content']) > 20):
                        key_images_info.append({
                            'index': global_index,
                            'file_path': file_path,
                            'analysis': analysis
                        })
                        print(f"  🔑 关键图像: 包含重要内容")
                
                print(f"  ✅ 分析完成 (耗时: {result.get('latency', {}).get('total_ms', 0):.2f}ms)")
                
            except Exception as e:
                print(f"❌ 处理图像失败: {str(e)}")
                batch_results.append({
                    'success': False,
                    'file_path': file_path,
                    'batch_index': global_index,
                    'error': str(e)
                })
            
            # 避免API限流
            time.sleep(1)
        
        results.extend(batch_results)
        print(f"✅ 批次 {batch_start//BATCH_SIZE + 1} 处理完成\n")
        
        # 批次间休息
        if batch_end < len(image_files):
            print("⏱️ 批次间休息 3 秒...")
            time.sleep(3)
    
    total_time = time.time() - total_start_time
    
    # 打印批量处理统计
    successful = [r for r in results if r.get('success', True)]
    failed = [r for r in results if not r.get('success', True)]
    
    print("📊 批量处理统计:")
    print(f"  • 处理目录: {directory_path}")
    print(f"  • 总图像数: {len(image_files)}")
    print(f"  • 成功处理: {len(successful)}")
    print(f"  • 处理失败: {len(failed)}")
    print(f"  • 关键图像: {len(key_images_info)}")
    print(f"  • 总耗时: {total_time:.2f}秒")
    
    if successful:
        avg_latency = sum(r.get('latency', {}).get('total_ms', 0) for r in successful) / len(successful)
        print(f"  • 平均延迟: {avg_latency:.2f}ms")
    
    # 保存关键图像信息
    if key_images_info:
        key_images_file = os.path.join('temp', 'key_images_info.json')
        with open(key_images_file, 'w', encoding='utf-8') as f:
            json.dump(key_images_info, f, ensure_ascii=False, indent=2)
        print(f"  • 关键图像信息已保存: {key_images_file}")
    
    return results

print("✅ 批量图像处理功能已定义")

## 5. ComfyUI 图生视频功能

In [ ]:
def image_to_base64(image_path: str) -> str:
    """
    将图像文件转换为Base64编码
    """
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
            base64_encoded = base64.b64encode(image_data).decode('utf-8')
            return base64_encoded
    except IOError:
        print(f"无法读取文件: {image_path}")
        return None

def queue_prompt(prompt, server_url):
    """
    向ComfyUI服务器提交任务
    """
    client_id = str(uuid.uuid4())
    p = {"prompt": prompt, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    url = f"http://{server_url}/prompt"
    req = urllib.request.Request(url, data=data)
    return json.loads(urllib.request.urlopen(req).read())

def get_video_by_prompt_id(prompt_id, server_url):
    """
    根据prompt_id获取生成的视频
    """
    def get_history(prompt_id):
        with urllib.request.urlopen(f"http://{server_url}/history/{prompt_id}") as response:
            return json.loads(response.read())
    
    def get_video(filename, subfolder, folder_type):
        data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
        url_values = urllib.parse.urlencode(data)
        with urllib.request.urlopen(f"http://{server_url}/view?{url_values}") as response:
            return response.read()
    
    output_videos = {}
    while True:
        try:
            history = get_history(prompt_id)[prompt_id]
            for node_id in history['outputs']:
                node_output = history['outputs'][node_id]
                # 视频输出分支
                if 'gifs' in node_output:
                    videos_output = []
                    for video in node_output['gifs']:
                        video_data = get_video(video['filename'], video['subfolder'], video['type'])
                        videos_output.append(video_data)
                    output_videos[node_id] = videos_output
            break
        except Exception as e:
            print(f"等待执行历史: {e}")
            time.sleep(5)
            continue
    
    return output_videos

def generate_video_from_image(image_path: str, video_prompt: str, output_path: str) -> bool:
    """
    使用ComfyUI从图像生成视频
    
    Args:
        image_path: 输入图像路径
        video_prompt: 视频生成提示词
        output_path: 输出视频路径
    
    Returns:
        是否成功生成视频
    """
    if not COMFYUI_SERVER_URL or not COMFYUI_WORKFLOW_PATH:
        print("❌ ComfyUI配置未完成，请设置COMFYUI_SERVER_URL和COMFYUI_WORKFLOW_PATH")
        return False
    
    try:
        # 读取workflow模板
        with open(COMFYUI_WORKFLOW_PATH, 'r') as f:
            workflow = json.load(f)
        
        # 将图像转换为base64
        base64_image = image_to_base64(image_path)
        if not base64_image:
            return False
        
        # 修改workflow中的参数（这里需要根据实际的workflow结构调整）
        # 假设workflow中有图像输入节点和文本提示节点
        # 用户需要根据实际workflow调整这些节点ID和参数名
        
        # 示例：设置图像输入（需要根据实际workflow调整）
        if 'image_input_node_id' in workflow:  # 替换为实际的节点ID
            workflow['image_input_node_id']['inputs']['image'] = base64_image
        
        # 示例：设置文本提示（需要根据实际workflow调整）
        if 'text_prompt_node_id' in workflow:  # 替换为实际的节点ID
            workflow['text_prompt_node_id']['inputs']['text'] = video_prompt
        
        print(f"🎬 开始生成视频: {os.path.basename(image_path)}")
        print(f"📝 视频提示词: {video_prompt}")
        
        # 提交任务
        result = queue_prompt(workflow, COMFYUI_SERVER_URL)
        prompt_id = result['prompt_id']
        print(f"📋 任务ID: {prompt_id}")
        
        # 获取生成的视频
        output_videos = get_video_by_prompt_id(prompt_id, COMFYUI_SERVER_URL)
        
        if output_videos:
            # 保存第一个生成的视频
            for node_id, videos in output_videos.items():
                if videos:
                    with open(output_path, 'wb') as f:
                        f.write(videos[0])
                    print(f"✅ 视频已保存: {output_path}")
                    return True
        
        print("❌ 未生成视频")
        return False
        
    except Exception as e:
        print(f"❌ 视频生成失败: {str(e)}")
        return False

print("✅ ComfyUI 图生视频功能已定义")
print("⚠️ 注意：需要根据实际的ComfyUI workflow调整节点ID和参数")

## 6. GPT-SoVITS 语音生成功能

In [ ]:
def upsert(lst, new_dict):
    """
    更新或插入字典到列表中
    """
    for i, item in enumerate(lst):
        if new_dict['index'] == i:
            lst[i] = new_dict
            return lst
    lst.append(new_dict)
    return lst

def invoke_gpt_sovits_endpoint(smr_client, endpoint_name, request):
    """
    调用GPT-SoVITS端点生成语音
    """
    content_type = "application/json"
    payload = json.dumps(request, ensure_ascii=False)

    response_model = smr_client.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        ContentType=content_type,
        Body=payload,
    )

    result = []
    print(f"📡 响应元数据: {response_model['ResponseMetadata']}")
    event_stream = iter(response_model['Body'])
    index = 0
    chunk_bytes = None
    
    try: 
        while True:
            event = next(event_stream)
            eventChunk = event['PayloadPart']['Bytes']
            chunk_dict = {}
            if index == 0:
                print("📦 收到第一个音频块")
                chunk_dict['first_chunk'] = True
                chunk_dict['bytes'] = eventChunk
                chunk_bytes = eventChunk
                chunk_dict['last_chunk'] = False
                chunk_dict['index'] = index
            else:
                chunk_dict['first_chunk'] = False
                chunk_dict['bytes'] = eventChunk
                chunk_bytes = eventChunk
                chunk_dict['last_chunk'] = False
                chunk_dict['index'] = index
            print(f"📦 音频块长度: {len(chunk_dict['bytes'])}")
            result.append(chunk_dict)    
            index += 1
    except StopIteration:
        print("✅ 所有音频块处理完成")
        chunk_dict = {}
        chunk_dict['first_chunk'] = False
        chunk_dict['bytes'] = chunk_bytes
        chunk_dict['last_chunk'] = True
        chunk_dict['index'] = index-1
        result = upsert(result, chunk_dict)
    
    return result

def generate_audio_from_text(output_path: str, prompt_text: str = None) -> bool:
    """
    使用GPT-SoVITS从文本生成语音
    
    Args:
        text: 要转换为语音的文本
        output_path: 输出音频文件路径
        prompt_text: 提示文本（可选）
    
    Returns:
        是否成功生成音频
    """
    if not GPT_SOVITS_ENDPOINT or not REFERENCE_AUDIO_PATH:
        print("❌ GPT-SoVITS配置未完成，请设置GPT_SOVITS_ENDPOINT和REFERENCE_AUDIO_PATH")
        return False
    
    try:
        runtime_sm_client = boto3.client(service_name="sagemaker-runtime")
        
        
        # 构建请求数据
        data = {
            "text": REFERENCE_TEXT,
            "text_lang": "zh",
            "ref_audio_path": REFERENCE_AUDIO_PATH,
            "prompt_lang": "zh",
            "prompt_text": prompt_text,
            "top_k": 5,
            "top_p": 1.0,
            "temperature": 0.7,
            "text_split_method": "cut5",
            "batch_size": 1,
            "batch_threshold": 0.75,
            "split_bucket": True,
            "speed_factor": 1.0,
            "fragment_interval": 0.3,
            "seed": -1,
            "media_type": "wav",
            "streaming_mode": False,
            "parallel_infer": True,
            "repetition_penalty": 1.35,
            "sample_steps": 32,
            "super_sampling": False
        }
        
        print(f"🔊 开始生成语音")
        print(f"📝 文本内容: {text[:100]}{'...' if len(text) > 100 else ''}")
        
        # 调用GPT-SoVITS端点
        response = invoke_gpt_sovits_endpoint(runtime_sm_client, GPT_SOVITS_ENDPOINT, data)
        
        # 合并音频数据
        audio_data = b''.join(chunk['bytes'] for chunk in response)
        
        # 保存音频文件
        with open(output_path, 'wb') as f:
            f.write(audio_data)
        
        print(f"✅ 音频已保存: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ 语音生成失败: {str(e)}")
        return False

print("✅ GPT-SoVITS 语音生成功能已定义")

## 7. 视频和音频合并功能

In [ ]:
def merge_video_audio(video_path: str, audio_path: str, output_path: str) -> bool:
    """
    合并视频和音频文件
    
    Args:
        video_path: 视频文件路径
        audio_path: 音频文件路径
        output_path: 输出文件路径
    
    Returns:
        是否成功合并
    """
    try:
        cmd = [
            'ffmpeg',
            '-i', video_path,
            '-i', audio_path,
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-strict', 'experimental',
            '-y',  # 覆盖输出文件
            output_path
        ]
        
        print(f"🎞️ 合并视频和音频")
        print(f"📹 视频: {os.path.basename(video_path)}")
        print(f"🔊 音频: {os.path.basename(audio_path)}")
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ 合并完成: {output_path}")
            return True
        else:
            print(f"❌ 合并失败: {result.stderr}")
            return False
            
    except Exception as e:
        print(f"❌ 合并过程出错: {str(e)}")
        return False

def concatenate_videos(video_paths: List[str], output_path: str) -> bool:
    """
    拼接多个视频文件
    
    Args:
        video_paths: 视频文件路径列表
        output_path: 输出文件路径
    
    Returns:
        是否成功拼接
    """
    if not video_paths:
        print("❌ 没有视频文件需要拼接")
        return False
    
    if len(video_paths) == 1:
        # 只有一个视频，直接复制
        try:
            import shutil
            shutil.copy2(video_paths[0], output_path)
            print(f"✅ 单个视频已复制: {output_path}")
            return True
        except Exception as e:
            print(f"❌ 复制视频失败: {str(e)}")
            return False
    
    try:
        # 创建临时文件列表
        temp_list_file = os.path.join('temp', 'video_list.txt')
        with open(temp_list_file, 'w') as f:
            for video_path in video_paths:
                # 使用绝对路径避免路径问题
                abs_path = os.path.abspath(video_path)
                f.write(f"file '{abs_path}'\n")
        
        cmd = [
            'ffmpeg',
            '-f', 'concat',
            '-safe', '0',
            '-i', temp_list_file,
            '-c', 'copy',
            '-y',  # 覆盖输出文件
            output_path
        ]
        
        print(f"🎬 拼接 {len(video_paths)} 个视频文件")
        for i, path in enumerate(video_paths, 1):
            print(f"  {i}. {os.path.basename(path)}")
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        # 清理临时文件
        if os.path.exists(temp_list_file):
            os.remove(temp_list_file)
        
        if result.returncode == 0:
            print(f"✅ 视频拼接完成: {output_path}")
            return True
        else:
            print(f"❌ 视频拼接失败: {result.stderr}")
            return False
            
    except Exception as e:
        print(f"❌ 拼接过程出错: {str(e)}")
        return False

def get_video_duration(video_path: str) -> float:
    """
    获取视频时长（秒）
    """
    try:
        cmd = [
            'ffprobe',
            '-v', 'quiet',
            '-print_format', 'json',
            '-show_format',
            video_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            info = json.loads(result.stdout)
            duration = float(info['format']['duration'])
            return duration
        else:
            print(f"❌ 获取视频时长失败: {result.stderr}")
            return 0.0
            
    except Exception as e:
        print(f"❌ 获取视频时长出错: {str(e)}")
        return 0.0

print("✅ 视频和音频合并功能已定义")

## 8. 主要处理流程

In [ ]:
def process_comic_to_video_voice(input_directory: str) -> str:
    """
    完整的漫画转视频配音流程
    
    Args:
        input_directory: 输入漫画图像目录
    
    Returns:
        最终输出视频路径
    """
    print("🚀 开始漫画转视频配音流程")
    print("="*60)
    
    # 步骤1: 批量分析漫画图像
    print("\n📊 步骤1: 分析漫画图像内容")
    analysis_results = batch_analyze_comic_images(input_directory, MAX_IMAGES)
    
    if not analysis_results:
        print("❌ 没有成功分析的图像")
        return None
    
    # 过滤成功的分析结果
    successful_results = [r for r in analysis_results if r.get('success')]
    if not successful_results:
        print("❌ 没有成功分析的图像")
        return None
    
    print(f"✅ 成功分析 {len(successful_results)} 张图像")
    
    # 步骤2: 生成视频
    print("\n🎬 步骤2: 生成视频")
    video_files = []
    audio_scripts = []
    
    for i, result in enumerate(successful_results):
        analysis = result['analysis_result']
        image_path = result['file_path']
        
        # 生成视频文件名
        video_filename = f"video_{i:03d}.mp4"
        video_path = os.path.join('output_videos', video_filename)
        
        # 获取视频提示词
        video_prompt = analysis.get('video_prompt', 'animate the comic scene')
        
        print(f"\n🎥 生成视频 {i+1}/{len(successful_results)}")
        if generate_video_from_image(image_path, video_prompt, video_path):
            video_files.append(video_path)
            # 收集音频脚本
            audio_script = analysis.get('audio_script', analysis.get('story_content', ''))
            if audio_script and audio_script.strip():
                audio_scripts.append(audio_script)
            else:
                audio_scripts.append(f"第{i+1}个场景")
        else:
            print(f"⚠️ 视频 {i+1} 生成失败，跳过")
    
    if not video_files:
        print("❌ 没有成功生成的视频")
        return None
    
    print(f"✅ 成功生成 {len(video_files)} 个视频")
    
    # 步骤3: 生成语音
    print("\n🔊 步骤3: 生成语音")
    audio_files = []
    
    for i, script in enumerate(audio_scripts):
        audio_filename = f"audio_{i:03d}.wav"
        audio_path = os.path.join('output_audio', audio_filename)
        
        print(f"\n🎙️ 生成语音 {i+1}/{len(audio_scripts)}")
        if generate_audio_from_text(output_path=audio_path,prompt_text=script):
            audio_files.append(audio_path)
        else:
            print(f"⚠️ 语音 {i+1} 生成失败，跳过")
            audio_files.append(None)
    
    print(f"✅ 成功生成 {len([a for a in audio_files if a])} 个语音文件")
    
    # 步骤4: 合并视频和音频
    print("\n🎞️ 步骤4: 合并视频和音频")
    final_video_files = []
    
    for i, (video_path, audio_path) in enumerate(zip(video_files, audio_files)):
        final_filename = f"final_{i:03d}.mp4"
        final_path = os.path.join('final_videos', final_filename)
        
        if audio_path and os.path.exists(audio_path):
            print(f"\n🎬 合并视频音频 {i+1}/{len(video_files)}")
            if merge_video_audio(video_path, audio_path, final_path):
                final_video_files.append(final_path)
            else:
                print(f"⚠️ 合并失败，使用原视频")
                final_video_files.append(video_path)
        else:
            print(f"⚠️ 没有对应音频，使用原视频")
            final_video_files.append(video_path)
    
    # 步骤5: 拼接所有视频
    print("\n🎬 步骤5: 拼接最终视频")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    final_output = f"comic_video_{timestamp}.mp4"
    
    if concatenate_videos(final_video_files, final_output):
        print(f"\n🎉 流程完成！")
        print(f"📁 最终视频: {final_output}")
        
        # 显示统计信息
        total_duration = sum(get_video_duration(vf) for vf in final_video_files if os.path.exists(vf))
        print(f"⏱️ 总时长: {total_duration:.2f} 秒")
        print(f"📊 处理统计:")
        print(f"  • 输入图像: {len(analysis_results)}")
        print(f"  • 生成视频: {len(video_files)}")
        print(f"  • 生成语音: {len([a for a in audio_files if a])}")
        
        return final_output
    else:
        print("❌ 最终视频拼接失败")
        return None

print("✅ 主要处理流程已定义")

## 9. 使用示例

In [ ]:
# 配置示例（用户需要根据实际情况修改）
print("📋 配置检查")
print("请确保以下配置已正确设置：")
print(f"• ComfyUI服务器: {COMFYUI_SERVER_URL or '未设置'}")
print(f"• ComfyUI工作流: {COMFYUI_WORKFLOW_PATH or '未设置'}")
print(f"• GPT-SoVITS端点: {GPT_SOVITS_ENDPOINT or '未设置'}")
print(f"• 参考音频路径: {REFERENCE_AUDIO_PATH or '未设置'}")
print()

# 示例配置（用户需要取消注释并填入实际值）
# COMFYUI_SERVER_URL = "your-comfyui-server.com:8080"
# COMFYUI_WORKFLOW_PATH = "path/to/your/workflow.json"
# GPT_SOVITS_ENDPOINT = "your-gpt-sovits-endpoint-name"
# REFERENCE_AUDIO_PATH = "s3://your-bucket/reference-audio.mp3"

print("📁 请将漫画图像放入 'input_images' 目录")
print("然后运行下面的代码开始处理")

In [ ]:
# 运行完整流程
# 注意：请确保已正确配置所有参数

input_dir = "input_images"

# 检查输入目录
if not os.path.exists(input_dir):
    print(f"❌ 输入目录不存在: {input_dir}")
    print("请创建目录并放入漫画图像")
else:
    images = get_comic_images_from_directory(input_dir)
    if not images:
        print(f"❌ 输入目录中没有图像文件: {input_dir}")
        print("请放入 .jpg, .jpeg, .png, .bmp, .gif 格式的图像")
    else:
        print(f"✅ 找到 {len(images)} 个图像文件")
        
        # 检查配置
        config_ok = True
        if not COMFYUI_SERVER_URL:
            print("❌ 请设置 COMFYUI_SERVER_URL")
            config_ok = False
        if not COMFYUI_WORKFLOW_PATH:
            print("❌ 请设置 COMFYUI_WORKFLOW_PATH")
            config_ok = False
        if not GPT_SOVITS_ENDPOINT:
            print("❌ 请设置 GPT_SOVITS_ENDPOINT")
            config_ok = False
        if not REFERENCE_AUDIO_PATH:
            print("❌ 请设置 REFERENCE_AUDIO_PATH")
            config_ok = False
        
        if config_ok:
            print("\n🚀 开始处理...")
            final_video = process_comic_to_video_voice(input_dir)
            
            if final_video:
                print(f"\n🎉 处理完成！")
                print(f"📹 最终视频: {final_video}")
            else:
                print("\n❌ 处理失败")
        else:
            print("\n⚠️ 请先完成配置再运行")

## 10. 工具函数和调试

In [ ]:
# 单独测试Bedrock图像分析
def test_image_analysis(image_path: str):
    """
    测试单张图像的分析功能
    """
    if not os.path.exists(image_path):
        print(f"❌ 图像文件不存在: {image_path}")
        return
    
    print(f"🔍 测试图像分析: {image_path}")
    result = analyze_comic_image_content(image_path)
    
    if result.get('success'):
        analysis = result['analysis_result']
        print("\n✅ 分析结果:")
        for key, value in analysis.items():
            print(f"  {key}: {value}")
    else:
        print(f"❌ 分析失败: {result.get('error')}")

# 清理临时文件
def cleanup_temp_files():
    """
    清理临时文件和目录
    """
    import shutil
    
    temp_dirs = ['temp', 'output_videos', 'output_audio']
    for temp_dir in temp_dirs:
        if os.path.exists(temp_dir):
            try:
                shutil.rmtree(temp_dir)
                os.makedirs(temp_dir, exist_ok=True)
                print(f"🧹 已清理: {temp_dir}")
            except Exception as e:
                print(f"❌ 清理失败 {temp_dir}: {e}")

# 检查依赖
def check_dependencies():
    """
    检查系统依赖
    """
    print("🔧 检查系统依赖:")
    
    # 检查ffmpeg
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("  ✅ ffmpeg 已安装")
        else:
            print("  ❌ ffmpeg 未正确安装")
    except FileNotFoundError:
        print("  ❌ ffmpeg 未找到")
    
    # 检查ffprobe
    try:
        result = subprocess.run(['ffprobe', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("  ✅ ffprobe 已安装")
        else:
            print("  ❌ ffprobe 未正确安装")
    except FileNotFoundError:
        print("  ❌ ffprobe 未找到")
    
    # 检查Python包
    required_packages = ['boto3', 'requests', 'pydub']
    for package in required_packages:
        try:
            __import__(package)
            print(f"  ✅ {package} 已安装")
        except ImportError:
            print(f"  ❌ {package} 未安装")

print("✅ 工具函数已定义")
print("\n🎯 使用说明:")
print("1. 运行 check_dependencies() 检查系统依赖")
print("2. 配置 ComfyUI 和 GPT-SoVITS 参数")
print("3. 将漫画图像放入 input_images 目录")
print("4. 运行主流程开始处理")
print("5. 使用 test_image_analysis() 测试单张图像")
print("6. 使用 cleanup_temp_files() 清理临时文件")